In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

import pandas as pd
import sklearn
import xgboost

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from src.data_utils import get_temporal_train_test_sets
from src.model_utils import fit_model_and_get_predictions
from src.eval_utils import performance_assessment, performance_assessment_model_collection
from src.features_utils import add_log_amount, add_categorical_feature_dummies, add_hour_of_day, add_day_of_week

In [2]:
df = pd.read_csv("../data/raw/paysim.csv")

In [3]:
(train_df, test_df) = get_temporal_train_test_sets(df, date_column="step", test_size=0.2)

In [4]:
train_df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,5.090096e+06,5.090096e+06,5.090096e+06,5.090096e+06,5.090096e+06,5.090096e+06,5.090096e+06,5.090096e+06
mean,1.943995e+02,1.813541e+05,8.419131e+05,8.637539e+05,1.070614e+06,1.200955e+06,7.777849e-04,5.893798e-07
std,1.051460e+02,6.402322e+05,2.921126e+06,2.957869e+06,3.153286e+06,3.494687e+06,2.787795e-02,7.677106e-04
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.350000e+02,1.328998e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.060000e+02,7.568529e+04,1.394900e+04,0.000000e+00,1.365108e+05,2.196950e+05,0.000000e+00,0.000000e+00
75%,2.810000e+02,2.097788e+05,1.076300e+05,1.457821e+05,9.462027e+05,1.119754e+06,0.000000e+00,0.000000e+00
max,3.550000e+02,9.244552e+07,4.381886e+07,4.368662e+07,3.555534e+08,3.555534e+08,1.000000e+00,1.000000e+00


In [5]:
test_df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,1.272519e+06,1.272519e+06,1.272519e+06,1.272519e+06,1.272519e+06,1.272519e+06,1.272519e+06,1.272519e+06
mean,4.393884e+02,1.738927e+05,8.017657e+05,8.205556e+05,1.221026e+06,1.321137e+06,3.342976e-03,1.021596e-05
std,9.515020e+01,4.284750e+05,2.752554e+06,2.784400e+06,4.240371e+06,4.316569e+06,5.772177e-02,3.196226e-03
min,3.550000e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,3.780000e+02,1.378492e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,3.990000e+02,7.176216e+04,1.483300e+04,0.000000e+00,1.179238e+05,1.958271e+05,0.000000e+00,0.000000e+00
75%,4.900000e+02,2.044740e+05,1.060610e+05,1.383365e+05,9.297747e+05,1.079217e+06,0.000000e+00,0.000000e+00
max,7.430000e+02,1.995378e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00


Average amount is similar although the training dataset has more variability. That is expected since there are also many more variables in that dataset.

In [6]:
train_df['isFraud'].value_counts(normalize=True)

isFraud
0    0.999222
1    0.000778
Name: proportion, dtype: float64

In [7]:
test_df['isFraud'].value_counts(normalize=True)

isFraud
0    0.996657
1    0.003343
Name: proportion, dtype: float64

## Baseline model

%-wise many more fraudulent transactions in the test dataset. That makes it harder for the model so no overestimation of the model predictive power is expected.

In [8]:
classifier = LogisticRegression(max_iter=1000)
input_features = ["amount"]

model_and_predict_baseline_logit = fit_model_and_get_predictions(classifier, train_df, test_df, input_features=input_features, output_feature="isFraud")

In [9]:
test_df["predicted_isFraud"] = model_and_predict_baseline_logit['predictions_test']

test_df.head(10)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,predicted_isFraud
5090096,355,CASH_OUT,129493.82,C1386423433,496957.0,367463.18,C1136528453,0.00,129493.82,0,0,0.000751
5090097,355,PAYMENT,5787.63,C1365884506,513408.0,507620.37,M1473495757,0.00,0.00,0,0,0.000737
5090098,355,CASH_OUT,172499.24,C468847406,150346.0,0.00,C1947417438,0.00,172499.24,0,0,0.000755
5090099,355,CASH_OUT,377391.01,C1354691509,30759.0,0.00,C1791454133,1409725.39,1697870.91,0,0,0.000778
5090100,355,PAYMENT,12469.50,C1287195798,315.0,0.00,M87390822,0.00,0.00,0,0,0.000738
5090101,355,PAYMENT,8180.47,C244693653,99055.0,90874.53,M689034153,0.00,0.00,0,0,0.000737
5090102,355,CASH_OUT,157613.10,C768128929,21221.0,0.00,C1185770817,0.00,157613.10,0,0,0.000754
5090103,355,PAYMENT,10845.92,C50339380,0.0,0.00,M543313363,0.00,0.00,0,0,0.000738
5090104,355,PAYMENT,4865.51,C122248941,22107.0,17241.49,M1867809818,0.00,0.00,0,0,0.000737
5090105,355,CASH_OUT,4100.58,C1822873210,10753.0,6652.42,C100403476,51525.03,55625.60,0,0,0.000737


In [10]:
performances_df = pd.DataFrame()
performances_df = performance_assessment(predictions_df=test_df, output_feature="isFraud", prediction_feature="predicted_isFraud")
performances_df.index = ["baseline"]

In [11]:
performances_df

,AUC ROC,Average precision
baseline,0.801,0.082


In [12]:
test_df["predicted_isFraud"].describe()

count    1.272519e+06
mean     7.573962e-04
std      7.771841e-05
min      7.365174e-04
25%      7.380002e-04
50%      7.442694e-04
75%      7.588207e-04
max      1.339459e-02
Name: predicted_isFraud, dtype: float64

The Fraud predictions are very conservative and close to the fraud rate in the dataset. 

In [13]:
test_df.groupby("isFraud")["predicted_isFraud"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,1268265.0,0.000757,0.000069,0.000737,0.000738,0.000744,0.000759,0.013395
1,4254.0,0.001016,0.000583,0.000737,0.000751,0.000788,0.000923,0.003164


The model does assign higher probabilities to the Fraudulent cases but the probabilities in general are very low.

Next steps:
- log transform the amount
- feature engineering for transaction type
- run logistic regression with more features

## Random classifier

In [14]:
predictions_df = test_df.copy()
predictions_df["predicted_isFraud"] = 0.5

performance_assessment(predictions_df=predictions_df, output_feature="isFraud", prediction_feature="predicted_isFraud")

,AUC ROC,Average precision
0,0.5,0.003


## Logistic regression with log amount

In [15]:
train_df_log = add_log_amount(train_df)
test_df_log = add_log_amount(test_df)

In [16]:
classifier = LogisticRegression(max_iter=1000)
input_features = ["log_amount"]

model_and_predict_log_logit = fit_model_and_get_predictions(classifier, train_df_log, test_df_log, input_features=input_features, output_feature="isFraud")

In [17]:
test_df_log["predicted_isFraud"] = model_and_predict_log_logit['predictions_test']
test_df_log.head(10)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,predicted_isFraud,log_amount
5090096,355,CASH_OUT,129493.82,C1386423433,496957.0,367463.18,C1136528453,0.00,129493.82,0,0,0.000698,11.771396
5090097,355,PAYMENT,5787.63,C1365884506,513408.0,507620.37,M1473495757,0.00,0.00,0,0,0.000055,8.663651
5090098,355,CASH_OUT,172499.24,C468847406,150346.0,0.00,C1947417438,0.00,172499.24,0,0,0.000882,12.058154
5090099,355,CASH_OUT,377391.01,C1354691509,30759.0,0.00,C1791454133,1409725.39,1697870.91,0,0,0.001668,12.841040
5090100,355,PAYMENT,12469.50,C1287195798,315.0,0.00,M87390822,0.00,0.00,0,0,0.000104,9.431121
5090101,355,PAYMENT,8180.47,C244693653,99055.0,90874.53,M689034153,0.00,0.00,0,0,0.000074,9.009627
5090102,355,CASH_OUT,157613.10,C768128929,21221.0,0.00,C1185770817,0.00,157613.10,0,0,0.000819,11.967905
5090103,355,PAYMENT,10845.92,C50339380,0.0,0.00,M543313363,0.00,0.00,0,0,0.000093,9.291636
5090104,355,PAYMENT,4865.51,C122248941,22107.0,17241.49,M1867809818,0.00,0.00,0,0,0.000048,8.490132
5090105,355,CASH_OUT,4100.58,C1822873210,10753.0,6652.42,C100403476,51525.03,55625.60,0,0,0.000042,8.319128


In [18]:
performance_log = performance_assessment(predictions_df=test_df_log, output_feature="isFraud", prediction_feature="predicted_isFraud")
performance_log.index = ["log_amount"]

In [19]:
performances_df = pd.concat([performances_df, performance_log])
performances_df

,AUC ROC,Average precision
baseline,0.801,0.082
log_amount,0.801,0.082


In [20]:
test_df_log["predicted_isFraud"].describe()

count    1.272519e+06
mean     7.587939e-04
std      1.255314e-03
min      4.762301e-08
25%      1.125564e-04
50%      4.316628e-04
75%      1.012745e-03
max      4.066680e-02
Name: predicted_isFraud, dtype: float64

Using log amount does not seem to have impact on model outcomes in a simple model

## Logistic regression with transaction types and amount

In [21]:
train_df_types = add_categorical_feature_dummies(train_df, column_name="type")
test_df_types = add_categorical_feature_dummies(test_df, column_name="type")

train_df_types = add_log_amount(train_df_types)
test_df_types = add_log_amount(test_df_types)

train_df_types.head(10)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,log_amount,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.0,0.00,0,0,9.194276,0.0,0.0,0.0,1.0,0.0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.0,0.00,0,0,7.531166,0.0,0.0,0.0,1.0,0.0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.0,0.00,1,0,5.204007,0.0,0.0,0.0,0.0,1.0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.0,0.00,1,0,5.204007,0.0,1.0,0.0,0.0,0.0
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.0,0.00,0,0,9.364703,0.0,0.0,0.0,1.0,0.0
5,1,PAYMENT,7817.71,C90045638,53860.00,46042.29,M573487274,0.0,0.00,0,0,8.964275,0.0,0.0,0.0,1.0,0.0
6,1,PAYMENT,7107.77,C154988899,183195.00,176087.23,M408069119,0.0,0.00,0,0,8.869085,0.0,0.0,0.0,1.0,0.0
7,1,PAYMENT,7861.64,C1912850431,176087.23,168225.59,M633326333,0.0,0.00,0,0,8.969878,0.0,0.0,0.0,1.0,0.0
8,1,PAYMENT,4024.36,C1265012928,2671.00,0.00,M1176932104,0.0,0.00,0,0,8.300370,0.0,0.0,0.0,1.0,0.0
9,1,DEBIT,5337.77,C712410124,41720.00,36382.23,C195600860,41898.0,40348.79,0,0,8.582751,0.0,0.0,1.0,0.0,0.0


In [22]:
classifier = LogisticRegression(max_iter=1000)
input_features = ["log_amount", "type_CASH_OUT", "type_CASH_IN", "type_DEBIT", "type_PAYMENT", "type_TRANSFER"]

model_and_predict_type_logit = fit_model_and_get_predictions(classifier, train_df_types, test_df_types, input_features=input_features, output_feature="isFraud")

test_df_types["predicted_isFraud"] = model_and_predict_type_logit['predictions_test']
test_df_types.head(10)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,predicted_isFraud,log_amount,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
5090096,355,CASH_OUT,129493.82,C1386423433,496957.0,367463.18,C1136528453,0.00,129493.82,0,0,1.145600e-03,11.771396,0.0,1.0,0.0,0.0,0.0
5090097,355,PAYMENT,5787.63,C1365884506,513408.0,507620.37,M1473495757,0.00,0.00,0,0,8.694709e-17,8.663651,0.0,0.0,0.0,1.0,0.0
5090098,355,CASH_OUT,172499.24,C468847406,150346.0,0.00,C1947417438,0.00,172499.24,0,0,1.213797e-03,12.058154,0.0,1.0,0.0,0.0,0.0
5090099,355,CASH_OUT,377391.01,C1354691509,30759.0,0.00,C1791454133,1409725.39,1697870.91,0,0,1.421346e-03,12.841040,0.0,1.0,0.0,0.0,0.0
5090100,355,PAYMENT,12469.50,C1287195798,315.0,0.00,M87390822,0.00,0.00,0,0,1.015190e-16,9.431121,0.0,0.0,0.0,1.0,0.0
5090101,355,PAYMENT,8180.47,C244693653,99055.0,90874.53,M689034153,0.00,0.00,0,0,9.323745e-17,9.009627,0.0,0.0,0.0,1.0,0.0
5090102,355,CASH_OUT,157613.10,C768128929,21221.0,0.00,C1185770817,0.00,157613.10,0,0,1.191908e-03,11.967905,0.0,1.0,0.0,0.0,0.0
5090103,355,PAYMENT,10845.92,C50339380,0.0,0.00,M543313363,0.00,0.00,0,0,9.869998e-17,9.291636,0.0,0.0,0.0,1.0,0.0
5090104,355,PAYMENT,4865.51,C122248941,22107.0,17241.49,M1867809818,0.00,0.00,0,0,8.395390e-17,8.490132,0.0,0.0,0.0,1.0,0.0
5090105,355,CASH_OUT,4100.58,C1822873210,10753.0,6652.42,C100403476,51525.03,55625.60,0,0,5.709346e-04,8.319128,0.0,1.0,0.0,0.0,0.0


In [23]:
performance_types = performance_assessment(predictions_df=test_df_types, output_feature="isFraud", prediction_feature="predicted_isFraud")
performance_types.index = ["types_and_log_amount"]
performances_df = pd.concat([performances_df, performance_types])
performances_df


,AUC ROC,Average precision
baseline,0.801,0.082
log_amount,0.801,0.082
types_and_log_amount,0.892,0.038


Adding transaction types increased ROC but decreased average precision.

Next steps:
- run other models with log amount and transaction types
- add feature modelling 'step' - hour of the day and weekday - run different models with these

## Decision tree, random forest, and gradient boosting models

In [24]:
train_df_types = add_categorical_feature_dummies(train_df, column_name="type")
test_df_types = add_categorical_feature_dummies(test_df, column_name="type")

train_df_types = add_log_amount(train_df_types)
test_df_types = add_log_amount(test_df_types)

train_df_types.head(10)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,log_amount,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.0,0.00,0,0,9.194276,0.0,0.0,0.0,1.0,0.0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.0,0.00,0,0,7.531166,0.0,0.0,0.0,1.0,0.0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.0,0.00,1,0,5.204007,0.0,0.0,0.0,0.0,1.0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.0,0.00,1,0,5.204007,0.0,1.0,0.0,0.0,0.0
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.0,0.00,0,0,9.364703,0.0,0.0,0.0,1.0,0.0
5,1,PAYMENT,7817.71,C90045638,53860.00,46042.29,M573487274,0.0,0.00,0,0,8.964275,0.0,0.0,0.0,1.0,0.0
6,1,PAYMENT,7107.77,C154988899,183195.00,176087.23,M408069119,0.0,0.00,0,0,8.869085,0.0,0.0,0.0,1.0,0.0
7,1,PAYMENT,7861.64,C1912850431,176087.23,168225.59,M633326333,0.0,0.00,0,0,8.969878,0.0,0.0,0.0,1.0,0.0
8,1,PAYMENT,4024.36,C1265012928,2671.00,0.00,M1176932104,0.0,0.00,0,0,8.300370,0.0,0.0,0.0,1.0,0.0
9,1,DEBIT,5337.77,C712410124,41720.00,36382.23,C195600860,41898.0,40348.79,0,0,8.582751,0.0,0.0,1.0,0.0,0.0


In [25]:
classifiers_dictionary={'tree_depth_3':DecisionTreeClassifier(max_depth=3,random_state=0), 
                        'tree_unlim_depth':DecisionTreeClassifier(random_state=0), 
                        'random_forest':RandomForestClassifier(random_state=0,n_jobs=-1),
                        'XGBoost':xgboost.XGBClassifier(random_state=0,n_jobs=-1),
                       }
input_features = ["log_amount", "type_CASH_OUT", "type_CASH_IN", "type_DEBIT", "type_PAYMENT", "type_TRANSFER"]

fitted_models_and_predictions_dictionary={}

for classifier_name in classifiers_dictionary:
    model_and_predictions = fit_model_and_get_predictions(classifiers_dictionary[classifier_name], train_df_types, test_df_types, input_features=input_features, output_feature="isFraud")
    
    fitted_models_and_predictions_dictionary[classifier_name] = model_and_predictions

In [26]:
performances_df = performance_assessment_model_collection(fitted_models_and_predictions_dictionary, transactions_df = test_df_types, type_set='test', output_feature="isFraud")
performances_df

,AUC ROC,Average precision,training_execution_time,prediction_execution_time
tree_depth_3,0.806,0.195,7.423669,0.068965
tree_unlim_depth,0.581,0.076,34.114331,0.178950
random_forest,0.612,0.159,439.985485,3.210983
XGBoost,0.920,0.241,13.328461,0.704098


XGBoost performs by far the best of all algorithms followed by a tree of depth three. Random forest is by far the slowest.

Next step at some point: find optimal tree depth.

## Add step-derived features

In [39]:
train_df_step = add_categorical_feature_dummies(train_df, column_name="type")
test_df_step= add_categorical_feature_dummies(test_df, column_name="type")

train_df_step = add_log_amount(train_df_step)
test_df_step = add_log_amount(test_df_step)

train_df_step = add_hour_of_day(train_df_step, column_name="step")
train_df_step = add_categorical_feature_dummies(train_df_step, column_name="hour_of_day")
test_df_step = add_hour_of_day(test_df_step, column_name="step")
test_df_step = add_categorical_feature_dummies(test_df_step, column_name="hour_of_day")

train_df_step = add_day_of_week(train_df_step, column_name="step")
train_df_step = add_categorical_feature_dummies(train_df_step, column_name="day_of_week")
test_df_step = add_day_of_week(test_df_step, column_name="step")
test_df_step = add_categorical_feature_dummies(test_df_step, column_name="day_of_week")

train_df_step.head(10)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,log_amount,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,hour_of_day,hour_of_day_0,hour_of_day_1,hour_of_day_2,hour_of_day_3,hour_of_day_4,hour_of_day_5,hour_of_day_6,hour_of_day_7,hour_of_day_8,hour_of_day_9,hour_of_day_10,hour_of_day_11,hour_of_day_12,hour_of_day_13,hour_of_day_14,hour_of_day_15,hour_of_day_16,hour_of_day_17,hour_of_day_18,hour_of_day_19,hour_of_day_20,hour_of_day_21,hour_of_day_22,hour_of_day_23,day_of_week,day_of_week_0,day_of_week_1,day_of_week_2,day_of_week_3,day_of_week_4,day_of_week_5,day_of_week_6
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.0,0.00,0,0,9.194276,0.0,0.0,0.0,1.0,0.0,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.0,0.00,0,0,7.531166,0.0,0.0,0.0,1.0,0.0,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.0,0.00,1,0,5.204007,0.0,0.0,0.0,0.0,1.0,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.0,0.00,1,0,5.204007,0.0,1.0,0.0,0.0,0.0,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.0,0.00,0,0,9.364703,0.0,0.0,0.0,1.0,0.0,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
5,1,PAYMENT,7817.71,C90045638,53860.00,46042.29,M573487274,0.0,0.00,0,0,8.964275,0.0,0.0,0.0,1.0,0.0,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
6,1,PAYMENT,7107.77,C154988899,183195.00,176087.23,M408069119,0.0,0.00,0,0,8.869085,0.0,0.0,0.0,1.0,0.0,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
7,1,PAYMENT,7861.64,C1912850431,176087.23,168225.59,M633326333,0.0,0.00,0,0,8.969878,0.0,0.0,0.0,1.0,0.0,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
8,1,PAYMENT,4024.36,C1265012928,2671.00,0.00,M1176932104,0.0,0.00,0,0,8.300370,0.0,0.0,0.0,1.0,0.0,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
9,1,DEBIT,5337.77,C712410124,41720.00,36382.23,C195600860,41898.0,40348.79,0,0,8.582751,0.0,0.0,1.0,0.0,0.0,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [40]:
classifiers_dictionary={'logistic_regression':LogisticRegression(max_iter=1000,random_state=0),
                        'tree_depth_3':DecisionTreeClassifier(max_depth=3,random_state=0), 
                        'tree_unlim_depth':DecisionTreeClassifier(random_state=0), 
                        'random_forest':RandomForestClassifier(random_state=0,n_jobs=-1),
                        'XGBoost':xgboost.XGBClassifier(random_state=0,n_jobs=-1),
                       }
hour_of_day_columns = [col for col in train_df_step.columns if col.startswith("hour_of_day_")]
day_of_week_columns = [col for col in train_df_step.columns if col.startswith("day_of_week_")]
input_features = ["log_amount", "type_CASH_OUT", "type_CASH_IN", "type_DEBIT", "type_PAYMENT", "type_TRANSFER"] + hour_of_day_columns + day_of_week_columns

fitted_models_and_predictions_dictionary={}

for classifier_name in classifiers_dictionary:
    model_and_predictions = fit_model_and_get_predictions(classifiers_dictionary[classifier_name], train_df_step, test_df_step, input_features=input_features, output_feature="isFraud")
    
    fitted_models_and_predictions_dictionary[classifier_name] = model_and_predictions

In [41]:
performances_df_step = performance_assessment_model_collection(fitted_models_and_predictions_dictionary, transactions_df = test_df_step, type_set='test', output_feature="isFraud")
performances_df_step

,AUC ROC,Average precision,training_execution_time,prediction_execution_time
logistic_regression,0.915,0.253,18.035989,0.472819
tree_depth_3,0.565,0.041,13.529402,0.454391
tree_unlim_depth,0.686,0.067,67.212604,0.502904
random_forest,0.734,0.134,321.904062,2.722058
XGBoost,0.767,0.225,17.901075,0.745484


Addition of step-derived variables decreased performance of almost all of the models except for logistic regression which now outperforms XGBoost without step-derived features. 

Next steps:
- try dimensionality reduction techniques
- check model performances with only hour or only weekdays
- add more features - behavioural for transaction receiver
- isMerchant for receiver

## Step-derived hours only

In [42]:
train_df_hour = add_categorical_feature_dummies(train_df, column_name="type")
test_df_hour= add_categorical_feature_dummies(test_df, column_name="type")

train_df_hour = add_log_amount(train_df_hour)
test_df_hour = add_log_amount(test_df_hour)

train_df_hour = add_hour_of_day(train_df_hour, column_name="step")
train_df_hour = add_categorical_feature_dummies(train_df_hour, column_name="hour_of_day")
test_df_hour = add_hour_of_day(test_df_hour, column_name="step")
test_df_hour = add_categorical_feature_dummies(test_df_hour, column_name="hour_of_day")

train_df_hour.head(10)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,log_amount,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,hour_of_day,hour_of_day_0,hour_of_day_1,hour_of_day_2,hour_of_day_3,hour_of_day_4,hour_of_day_5,hour_of_day_6,hour_of_day_7,hour_of_day_8,hour_of_day_9,hour_of_day_10,hour_of_day_11,hour_of_day_12,hour_of_day_13,hour_of_day_14,hour_of_day_15,hour_of_day_16,hour_of_day_17,hour_of_day_18,hour_of_day_19,hour_of_day_20,hour_of_day_21,hour_of_day_22,hour_of_day_23
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.0,0.00,0,0,9.194276,0.0,0.0,0.0,1.0,0.0,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.0,0.00,0,0,7.531166,0.0,0.0,0.0,1.0,0.0,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.0,0.00,1,0,5.204007,0.0,0.0,0.0,0.0,1.0,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.0,0.00,1,0,5.204007,0.0,1.0,0.0,0.0,0.0,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.0,0.00,0,0,9.364703,0.0,0.0,0.0,1.0,0.0,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,1,PAYMENT,7817.71,C90045638,53860.00,46042.29,M573487274,0.0,0.00,0,0,8.964275,0.0,0.0,0.0,1.0,0.0,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,1,PAYMENT,7107.77,C154988899,183195.00,176087.23,M408069119,0.0,0.00,0,0,8.869085,0.0,0.0,0.0,1.0,0.0,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,1,PAYMENT,7861.64,C1912850431,176087.23,168225.59,M633326333,0.0,0.00,0,0,8.969878,0.0,0.0,0.0,1.0,0.0,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,1,PAYMENT,4024.36,C1265012928,2671.00,0.00,M1176932104,0.0,0.00,0,0,8.300370,0.0,0.0,0.0,1.0,0.0,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,1,DEBIT,5337.77,C712410124,41720.00,36382.23,C195600860,41898.0,40348.79,0,0,8.582751,0.0,0.0,1.0,0.0,0.0,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [43]:
classifiers_dictionary={'logistic_regression':LogisticRegression(max_iter=1000,random_state=0),
                        'tree_depth_3':DecisionTreeClassifier(max_depth=3,random_state=0), 
                        'tree_unlim_depth':DecisionTreeClassifier(random_state=0), 
                        'random_forest':RandomForestClassifier(random_state=0,n_jobs=-1),
                        'XGBoost':xgboost.XGBClassifier(random_state=0,n_jobs=-1),
                       }
hour_of_day_columns = [col for col in train_df_hour.columns if col.startswith("hour_of_day_")]
input_features = ["log_amount", "type_CASH_OUT", "type_CASH_IN", "type_DEBIT", "type_PAYMENT", "type_TRANSFER"] + hour_of_day_columns

fitted_models_and_predictions_dictionary={}

for classifier_name in classifiers_dictionary:
    model_and_predictions = fit_model_and_get_predictions(classifiers_dictionary[classifier_name], train_df_hour, test_df_hour, input_features=input_features, output_feature="isFraud")
    
    fitted_models_and_predictions_dictionary[classifier_name] = model_and_predictions

In [44]:
performances_df_hour = performance_assessment_model_collection(fitted_models_and_predictions_dictionary, transactions_df = test_df_hour, type_set='test', output_feature="isFraud")
performances_df_hour

,AUC ROC,Average precision,training_execution_time,prediction_execution_time
logistic_regression,0.924,0.250,17.653974,0.472116
tree_depth_3,0.565,0.070,11.284877,0.401088
tree_unlim_depth,0.640,0.155,57.326245,0.479003
random_forest,0.702,0.296,309.881869,2.815582
XGBoost,0.729,0.183,16.247103,0.687064


In [45]:
performances_df_step

,AUC ROC,Average precision,training_execution_time,prediction_execution_time
logistic_regression,0.915,0.253,18.035989,0.472819
tree_depth_3,0.565,0.041,13.529402,0.454391
tree_unlim_depth,0.686,0.067,67.212604,0.502904
random_forest,0.734,0.134,321.904062,2.722058
XGBoost,0.767,0.225,17.901075,0.745484


It seems that using only hour gives the best overall performance in combination with the random forest algorithm. However, most other models performed best (Looking at AP with both). 

## Step-derived days only

In [46]:
train_df_days = add_categorical_feature_dummies(train_df, column_name="type")
test_df_days= add_categorical_feature_dummies(test_df, column_name="type")

train_df_days = add_log_amount(train_df_days)
test_df_days = add_log_amount(test_df_days)

train_df_days = add_day_of_week(train_df_days, column_name="step")
train_df_days = add_categorical_feature_dummies(train_df_days, column_name="day_of_week")
test_df_days = add_day_of_week(test_df_days, column_name="step")
test_df_days = add_categorical_feature_dummies(test_df_days, column_name="day_of_week")

train_df_days.head(10)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,log_amount,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,day_of_week,day_of_week_0,day_of_week_1,day_of_week_2,day_of_week_3,day_of_week_4,day_of_week_5,day_of_week_6
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.0,0.00,0,0,9.194276,0.0,0.0,0.0,1.0,0.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.0,0.00,0,0,7.531166,0.0,0.0,0.0,1.0,0.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.0,0.00,1,0,5.204007,0.0,0.0,0.0,0.0,1.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.0,0.00,1,0,5.204007,0.0,1.0,0.0,0.0,0.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.0,0.00,0,0,9.364703,0.0,0.0,0.0,1.0,0.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
5,1,PAYMENT,7817.71,C90045638,53860.00,46042.29,M573487274,0.0,0.00,0,0,8.964275,0.0,0.0,0.0,1.0,0.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
6,1,PAYMENT,7107.77,C154988899,183195.00,176087.23,M408069119,0.0,0.00,0,0,8.869085,0.0,0.0,0.0,1.0,0.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
7,1,PAYMENT,7861.64,C1912850431,176087.23,168225.59,M633326333,0.0,0.00,0,0,8.969878,0.0,0.0,0.0,1.0,0.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
8,1,PAYMENT,4024.36,C1265012928,2671.00,0.00,M1176932104,0.0,0.00,0,0,8.300370,0.0,0.0,0.0,1.0,0.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
9,1,DEBIT,5337.77,C712410124,41720.00,36382.23,C195600860,41898.0,40348.79,0,0,8.582751,0.0,0.0,1.0,0.0,0.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [47]:
classifiers_dictionary={'logistic_regression':LogisticRegression(max_iter=1000,random_state=0),
                        'tree_depth_3':DecisionTreeClassifier(max_depth=3,random_state=0), 
                        'tree_unlim_depth':DecisionTreeClassifier(random_state=0), 
                        'random_forest':RandomForestClassifier(random_state=0,n_jobs=-1),
                        'XGBoost':xgboost.XGBClassifier(random_state=0,n_jobs=-1),
                       }
day_of_week_columns = [col for col in train_df_days.columns if col.startswith("day_of_week_")]
input_features = ["log_amount", "type_CASH_OUT", "type_CASH_IN", "type_DEBIT", "type_PAYMENT", "type_TRANSFER"] + day_of_week_columns

fitted_models_and_predictions_dictionary_days={}

for classifier_name in classifiers_dictionary:
    model_and_predictions = fit_model_and_get_predictions(classifiers_dictionary[classifier_name], train_df_days, test_df_days, input_features=input_features, output_feature="isFraud")
    
    fitted_models_and_predictions_dictionary_days[classifier_name] = model_and_predictions

In [48]:
performances_df_days = performance_assessment_model_collection(fitted_models_and_predictions_dictionary_days, transactions_df = test_df_days, type_set='test', output_feature="isFraud")
performances_df_days

,AUC ROC,Average precision,training_execution_time,prediction_execution_time
logistic_regression,0.823,0.014,6.288356,0.154785
tree_depth_3,0.806,0.203,9.270152,0.156629
tree_unlim_depth,0.589,0.058,32.925438,0.212979
random_forest,0.618,0.144,248.386116,2.170830
XGBoost,0.779,0.150,12.353207,0.560508


In [49]:
performances_df_hour

,AUC ROC,Average precision,training_execution_time,prediction_execution_time
logistic_regression,0.924,0.250,17.653974,0.472116
tree_depth_3,0.565,0.070,11.284877,0.401088
tree_unlim_depth,0.640,0.155,57.326245,0.479003
random_forest,0.702,0.296,309.881869,2.815582
XGBoost,0.729,0.183,16.247103,0.687064


In [50]:
performances_df_step

,AUC ROC,Average precision,training_execution_time,prediction_execution_time
logistic_regression,0.915,0.253,18.035989,0.472819
tree_depth_3,0.565,0.041,13.529402,0.454391
tree_unlim_depth,0.686,0.067,67.212604,0.502904
random_forest,0.734,0.134,321.904062,2.722058
XGBoost,0.767,0.225,17.901075,0.745484


Just using days decreased the performance of many of the models, however it does seem to work better for the trees.

Next steps:
- implement train, validation and test splits. 
- hyperparameter tuning - learn for the best models
- potentially add more features